# 03. Funnel Analysis

In [ ]:
import pandas as pd
import os
import kagglehub
import numpy as np
import matplotlib.pyplot as plt

dataset_path = kagglehub.dataset_download('radistaleks/synthetic-bank-transactions')
categories    = pd.read_csv(os.path.join(dataset_path, 'categories.csv'))
clients       = pd.read_csv(os.path.join(dataset_path, 'clients.csv'))
subscriptions = pd.read_csv(os.path.join(dataset_path, 'subscriptions.csv'))
transactions  = pd.read_csv(os.path.join(dataset_path, 'transactions.csv'))

In [ ]:
clients['registration_date'] = pd.to_datetime(clients['registration_date'])
subscriptions['date_start']  = pd.to_datetime(subscriptions['date_start'])
subscriptions['date_end']    = pd.to_datetime(subscriptions['date_end'])
transactions['date']         = pd.to_datetime(transactions['date'], format='%Y-%m-%d %H:%M:%S')

clients = clients.fillna(0)
subscriptions['product_company'] = subscriptions['product_company'].fillna('Неизвестно')
transactions['product_company']  = transactions['product_company'].fillna('Неизвестно')

cat_map = dict(zip(categories['id'], categories['name']))
transactions['category_name'] = transactions['product_category'].map(cat_map)

N = len(clients)
active_subs = subscriptions[subscriptions['date_end'].isna()]

## 1. Продуктовая воронка

In [ ]:
funnel = pd.Series({
    'Все клиенты':          N,
    'Есть транзакции':      transactions['client_id'].nunique(),
    'Есть кредит':          (clients['credit'] == 1).sum(),
    'Есть депозит':         (clients['deposit'] == 1).sum(),
    'Есть подписка':        active_subs['client_id'].nunique(),
    'Музыкальная подписка': active_subs[active_subs['product_category'] == 4]['client_id'].nunique(),
})
funnel

In [ ]:
# конверсия между шагами
(funnel / N * 100).round(1)

In [ ]:
funnel.plot(kind='barh', figsize=(10, 5), title='Продуктовая воронка')

In [ ]:
# пересечения продуктов
credit_set  = set(clients[clients['credit']  == 1]['id'])
deposit_set = set(clients[clients['deposit'] == 1]['id'])
sub_set     = set(active_subs['client_id'])
txn_set     = set(transactions['client_id'])

combos = pd.Series({
    'Только транзакции':          len(txn_set - credit_set - deposit_set - sub_set),
    'Транзакции + кредит':        len((txn_set & credit_set) - deposit_set - sub_set),
    'Транзакции + депозит':       len((txn_set & deposit_set) - credit_set - sub_set),
    'Транзакции + подписка':      len((txn_set & sub_set) - credit_set - deposit_set),
    'Кредит + депозит':           len((txn_set & credit_set & deposit_set) - sub_set),
    'Кредит + подписка':          len((txn_set & credit_set & sub_set) - deposit_set),
    'Депозит + подписка':         len((txn_set & deposit_set & sub_set) - credit_set),
    'Все 4 продукта':             len(txn_set & credit_set & deposit_set & sub_set),
}).sort_values()
combos

In [ ]:
combos.plot(kind='barh', figsize=(11, 5), title='Клиенты по комбинациям продуктов')

## 2. Категорийная воронка

In [ ]:
cat_reach = transactions.groupby('category_name')['client_id'].nunique().sort_values(ascending=False)
cat_reach_pct = (cat_reach / N * 100).round(1)
cat_reach_pct

In [ ]:
cat_reach_pct.sort_values().plot(kind='barh', figsize=(12, 10), title='Проникновение категорий, % клиентов')

In [ ]:
# тиры охвата
pd.cut(
    cat_reach_pct,
    bins=[0, 20, 50, 80, 100],
    labels=['Нишевые <20%', 'Популярные 20-50%', 'Массовые 50-80%', 'Охватные >80%']
).value_counts()

## 3. Выживаемость подписок

In [ ]:
subs = subscriptions.copy()
subs['end_filled']    = subs['date_end'].fillna(pd.Timestamp('2020-12-31'))
subs['duration_days'] = (subs['end_filled'] - subs['date_start']).dt.days
subs['is_churned']    = subs['date_end'].notna().astype(int)

print('Всего подписок: ', len(subs))
print('Отменено:       ', subs['is_churned'].sum())
print('Churn rate:     ', f"{subs['is_churned'].mean()*100:.1f}%")
print('Median duration:', subs['duration_days'].median(), 'дней')

In [ ]:
milestones = [30, 60, 90, 180, 365, 547, 730, 1095, 1460]
labels     = ['1м', '2м', '3м', '6м', '1г', '1.5г', '2г', '3г', '4г']

survival = pd.Series(
    [(subs['duration_days'] >= d).sum() / len(subs) * 100 for d in milestones],
    index=labels
)
survival

In [ ]:
survival.plot(marker='o', figsize=(12, 5), title='Survival Curve: % активных подписок')

In [ ]:
# survival по музыкальным сервисам
music = subs[subs['product_category'] == 4]
fig, ax = plt.subplots(figsize=(12, 5))
for company in music['product_company'].value_counts().head(6).index:
    co = music[music['product_company'] == company]
    sr = [(co['duration_days'] >= d).sum() / len(co) * 100 for d in milestones]
    pd.Series(sr, index=labels).plot(ax=ax, marker='o', label=f'{company} (n={len(co)})')
ax.set_title('Survival Curve по музыкальным сервисам')
ax.set_ylabel('% активных')
ax.legend()
plt.tight_layout()

In [ ]:
# churn rate по сервисам
music_churn = (
    music.groupby('product_company')['is_churned']
    .agg(['sum', 'count', 'mean'])
    .rename(columns={'sum': 'churned', 'count': 'total', 'mean': 'churn_rate'})
    .sort_values('churn_rate', ascending=False)
)
music_churn['churn_rate'] = (music_churn['churn_rate'] * 100).round(1)
music_churn

In [ ]:
music_churn['churn_rate'].plot(kind='bar', figsize=(10, 4), title='Churn Rate по музыкальным сервисам, %', rot=20)

## 4. Воронка типов транзакций

In [ ]:
subtype_reach = transactions.groupby('subtype')['client_id'].nunique().sort_values(ascending=False)
(subtype_reach / N * 100).round(1)

In [ ]:
(subtype_reach / N * 100).plot(kind='bar', figsize=(10, 4), title='% клиентов по типу транзакций', rot=0)

In [ ]:
# среднее кол-во транзакций каждого типа на клиента
transactions.groupby(['client_id', 'subtype'])['amount'].count().groupby('subtype').mean().round(1).sort_values(ascending=False)